# Experiment: Naive Paraphrasers and False Positives
The goal of this experiment is to find out whether Naive Paraphrasers risk False Positives because they (hypothesis) know too much about the original text.

## Dataset
We construct a dataset of original texts with their LLM generated paraphrases. The dataset should include:
- Original text
- Paraphrased text
- Label indicating which paraphraser generated the paraphrase

The dataset contains only false samples, i.e. the original text and the paraphrased text are not from the same author.
The original texts should be from a variety of sources to ensure diversity in genres.


In [ ]:
import os
from pathlib import Path
import pandas as pd
from genai_detection.util import preprocess_text as _preprocess_text


category2directory = {  # value: directory name and one file name in the datasets folder
    "News": [
        "custom_texts",
        "cnn_040725",
    ],  # Dalai Lama; "cnn_230625"    # USA attacks Iran
    "Blog": ["blog_corpus", 27],
    "Gutenberg": ["gutenberg", "Othello_the_Moor_of_Venice_William_Shakespeare"],
    "Student Essay": [
        "student_essays/Intro2006",
        "Ass1/2006_AB4847",
    ],  # stream of consciousness essay: 18 years old girl who writes about her bipolar ex-boyfriend of 8 months and her new boyfriend who loves music like her but is not catholic; she wants to live in the present
}

dataset = pd.DataFrame(
    columns=[
        "category",
        "original_text",
        "original_file_name",
        "text_length",
        # "paraphrased_text",
        # "paraphraser",
    ]
)


def load_text(data_category, path, file_name):
    if data_category == "Blog":
        df = pd.read_csv(path / "blogtext.csv")
        return df.loc[file_name, "text"]
    return open(path / f"{file_name}.txt", encoding="utf-8").read()


for data_category in category2directory.keys():
    path2datasets = (
        Path(os.getcwd()).resolve().parent
        / "data"
        / "datasets"
        / category2directory[data_category][0]
    )
    dir_name, file_name = category2directory[data_category]
    text = " ".join(
        _preprocess_text(load_text(data_category, path2datasets, file_name)).split()[
            :1221
        ]
    )
    dataset = pd.concat(
        [
            dataset,
            pd.DataFrame(
                [
                    {
                        "category": data_category,
                        "original_text": text,
                        "original_file_name": file_name,
                        "text_length": len(text.split()),
                    }
                ]
            ),
        ]
    )
dataset.reset_index(drop=True, inplace=True)
display(dataset)

Paraphrasers

In [ ]:
import sys

from tqdm import tqdm
from genai_detection.config import CONFIG
from genai_detection.paraphrasing.paraphraser import (
    BulletPointParaphraser,
    OllamaParaphraser,
    T5ChatGPTParaphraser,
    T5GooglePAWSParaphraser,
    TaskParaphraser,
    TitleParaphraser,
    TopicParaphraser,
    TranslationParaphraser,
)

OLLAMA_VERSION = "zephyr:7b"  # "mistral:7b"  # "default:latest"
PROMPTS = [
    "Paraphrase the text above and output only the paraphrased version.",
    "For the text above: First, extract bullet points capturing the main ideas, then create a text based on these bullet points. Only output the final text (i.e. do not output the bullet points or any additional chain of thoughts).",
    "For the text above: Paraphrase the sentence by first identifying the main subject, verb, and object. Then find synonyms for each and construct a new sentence. Only output the final paraphrased sentence.",
    "For the text above: Paraphrase the sentence using the same tone as the original with approximately the same number of words.",
    "For the text above: Paraphrase this sentence. Do not change the meaning, but use different words and structure. Output only the paraphrased sentence.",
]
temperature = CONFIG.TEMPERATURE


def create_paraphrasers(model):
    return {
        "T5_ChatGPT": T5ChatGPTParaphraser(),
        "T5_Google_PAWS": T5GooglePAWSParaphraser(),
        "Ollama": model,
        "BulletPoint": BulletPointParaphraser(model, model),
        "Task": TaskParaphraser(model, model),
        "Topic": TopicParaphraser(model, model),
        "Title": TitleParaphraser(model, model),
        "Translation": TranslationParaphraser(model, model),
    }


new_rows = []
model = OllamaParaphraser(model_id=OLLAMA_VERSION)
paraphrasers = create_paraphrasers(model)
for i in tqdm(dataset.index, desc="Processing dataset"):
    text = dataset.loc[i, "original_text"]
    # print(
    #     f"Processing text from {dataset.loc[i, 'category']} category, file: {dataset.loc[i, 'original_file_name']}, length: {dataset.loc[i, 'text_length']} words"
    # )
    for prompt in PROMPTS:

        paraphrase_config = {
            "text": text,
            "n_responses": 1,
            "prompt": prompt,
            "temperature": temperature,
        }
        for paraphraser_name, paraphraser in paraphrasers.items():
            print(f"Paraphrasing with {paraphraser_name}...")
            paraphrase = _preprocess_text(
                paraphraser.paraphrase(**paraphrase_config)[0]
            )
            # paraphrase = "Ja " * 1000

            # Save a copy of the original row + new info
            new_row = dataset.loc[i].copy()
            new_row["prompt"] = prompt
            new_row["paraphraser"] = paraphraser_name
            new_row["paraphrased_text"] = paraphrase
            new_rows.append(new_row)

# Create a new DataFrame from the new rows
expanded_dataset = pd.DataFrame(new_rows)
expanded_dataset.reset_index(drop=True, inplace=True)
# (Optional) Save or display
display(expanded_dataset.head())
# expanded_dataset.to_csv("paraphrased_dataset.csv", index=False)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Processing text from News category, file: cnn_040725, length: 582 words
Paraphrasing with T5_ChatGPT...
Paraphrasing with T5_Google_PAWS...
Paraphrasing with T5_ChatGPT...
Paraphrasing with T5_Google_PAWS...
Processing text from Blog category, file: 27, length: 933 words
Paraphrasing with T5_ChatGPT...
Paraphrasing with T5_Google_PAWS...
Paraphrasing with T5_ChatGPT...
Paraphrasing with T5_Google_PAWS...
Processing text from Gutenberg category, file: Othello_the_Moor_of_Venice_William_Shakespeare, length: 1221 words
Paraphrasing with T5_ChatGPT...
Paraphrasing with T5_Google_PAWS...
Paraphrasing with T5_ChatGPT...
Paraphrasing with T5_Google_PAWS...
Processing text from Student Essay category, file: Ass1/2006_AB4847, length: 1221 words
Paraphrasing with T5_ChatGPT...
Paraphrasing with T5_Google_PAWS...
Paraphrasing with T5_ChatGPT...
Paraphrasing with T5_Google_PAWS...


,category,original_text,original_file_name,text_length,prompt,paraphraser,paraphrased_text
0,News,"for much of the past century, the dalai lama h...",cnn_040725,582,Paraphrase the text above and output only the ...,T5_ChatGPT,Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja J...
1,News,"for much of the past century, the dalai lama h...",cnn_040725,582,Paraphrase the text above and output only the ...,T5_Google_PAWS,Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja J...
2,News,"for much of the past century, the dalai lama h...",cnn_040725,582,"For the text above: First, extract bullet poin...",T5_ChatGPT,Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja J...
3,News,"for much of the past century, the dalai lama h...",cnn_040725,582,"For the text above: First, extract bullet poin...",T5_Google_PAWS,Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja J...
4,Blog,this may be a long blog...got a lot of thought...,27,933,Paraphrase the text above and output only the ...,T5_ChatGPT,Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja J...


## AV Models
Multiple AV models should be used to evaluate the paraphrased texts.

In [ ]:
import numpy as np
from tqdm import tqdm
from genai_detection.detectors.impostor import ImpostorDetector
from genai_detection.detectors.unmasking import UnmaskingDetector
from genai_detection.detectors.ppmd import PPMdDetector

imposter_detector = ImpostorDetector(
    path2imp=Path(os.getcwd()).resolve().parent / CONFIG.PATH2BLOG
)
unmasking_detector = UnmaskingDetector()
ppmd_detector = PPMdDetector()


for detector in [imposter_detector, unmasking_detector, ppmd_detector]:
    expanded_dataset[f"{detector.__class__.__name__}_score"] = np.nan
    for i in tqdm(
        expanded_dataset.index, desc=f"Processing {detector.__class__.__name__} scores"
    ):
        original_text = expanded_dataset.loc[i, "original_text"]
        paraphrased_text = expanded_dataset.loc[i, "paraphrased_text"]
        score = detector.get_score([original_text, paraphrased_text], normalize=False)
        expanded_dataset.loc[i, f"{detector.__class__.__name__}_score"] = np.round(
            score, 2
        )

Processing PPMdDetector scores: 100%|██████████| 16/16 [00:00<00:00, 399.89it/s]


In [ ]:
save_path = (
    Path(os.getcwd()).resolve().parent / "results" / "paraphrasing" / "experiments"
)
if not save_path.exists():
    save_path.mkdir(parents=True, exist_ok=True)
display(expanded_dataset.head())
expanded_dataset.to_csv(
    save_path / "naive_paraphrase_av_scores_dataset.csv", index=False
)

Saving dataset to /Users/klara/Developer/Uni/MA/artificial-authorship-verification/results/paraphrasing/experiments


,category,original_text,original_file_name,text_length,prompt,paraphraser,paraphrased_text,UnmaskingDetector_score,PPMdDetector_score,ImpostorDetector_score
0,News,"for much of the past century, the dalai lama h...",cnn_040725,582,Paraphrase the text above and output only the ...,T5_ChatGPT,Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja J...,0.38,0.02,0.0
1,News,"for much of the past century, the dalai lama h...",cnn_040725,582,Paraphrase the text above and output only the ...,T5_Google_PAWS,Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja J...,0.38,0.02,0.0
2,News,"for much of the past century, the dalai lama h...",cnn_040725,582,"For the text above: First, extract bullet poin...",T5_ChatGPT,Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja J...,0.38,0.02,0.0
3,News,"for much of the past century, the dalai lama h...",cnn_040725,582,"For the text above: First, extract bullet poin...",T5_Google_PAWS,Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja J...,0.38,0.02,0.0
4,Blog,this may be a long blog...got a lot of thought...,27,933,Paraphrase the text above and output only the ...,T5_ChatGPT,Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja J...,0.38,0.02,0.0
5,Blog,this may be a long blog...got a lot of thought...,27,933,Paraphrase the text above and output only the ...,T5_Google_PAWS,Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja J...,0.38,0.02,0.0
6,Blog,this may be a long blog...got a lot of thought...,27,933,"For the text above: First, extract bullet poin...",T5_ChatGPT,Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja J...,0.38,0.02,0.0
7,Blog,this may be a long blog...got a lot of thought...,27,933,"For the text above: First, extract bullet poin...",T5_Google_PAWS,Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja J...,0.38,0.02,0.0
8,Gutenberg,venice. a street. enter roderigo and iago. tus...,Othello_the_Moor_of_Venice_William_Shakespeare,1221,Paraphrase the text above and output only the ...,T5_ChatGPT,Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja J...,0.38,0.02,0.0
9,Gutenberg,venice. a street. enter roderigo and iago. tus...,Othello_the_Moor_of_Venice_William_Shakespeare,1221,Paraphrase the text above and output only the ...,T5_Google_PAWS,Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja Ja J...,0.38,0.02,0.0


In [ ]:
# calculate rmse per paraphraser and AV model
rmse_per_paraphraser = {}
for paraphraser in expanded_dataset["paraphraser"].unique():
    rmse_per_paraphraser[paraphraser] = {}
    for detector in [imposter_detector, unmasking_detector, ppmd_detector]:
        scores = expanded_dataset[expanded_dataset["paraphraser"] == paraphraser][
            f"{detector.__class__.__name__}_score"
        ]
        rmse = np.sqrt(np.mean((scores - 0) ** 2))
        rmse_per_paraphraser[paraphraser][detector.__class__.__name__] = rmse
# to dataframe
rmse_df = pd.DataFrame(rmse_per_paraphraser).T
rmse_df.reset_index(inplace=True)
rmse_df.rename(columns={"index": "paraphraser"}, inplace=True)
# rmse_df.to_csv(save_path / "naive_paraphrase_rmse_per_paraphraser.csv", index=False)
print("RMSE per paraphraser:")
display(rmse_df)

RMSE per paraphraser:


,paraphraser,ImpostorDetector,UnmaskingDetector,PPMdDetector
0,T5_ChatGPT,0.0,0.38,0.02
1,T5_Google_PAWS,0.0,0.38,0.02
